In [ ]:
from pymongo import MongoClient
from urllib.parse import quote_plus
from PIL import Image
import os
from uuid import uuid4
import json
from tqdm.auto import tqdm

In [ ]:
DB_HOST = "127.0.0.1"
DB_USER = "trocr_client"
DB_PWD = "ji8zmA@JGoXAR>mW9gk*"
DB_SCHEMA = "trocr_data"
DB_PORT = ":27018"

In [ ]:
BASE_PATH = "/home/ralvarez22/Documentos/trocr_hand/trocr_api/static/5887483f4d92"
EXPORT_PATH = "/home/ralvarez22/Documentos/trocr_hand/trocr_llm/datasets/trocr_app_dataset"

In [ ]:
os.makedirs(EXPORT_PATH, exist_ok=True)

In [ ]:
conn_url = "mongodb://{}:{}@{}{}/{}".format(
    quote_plus(DB_USER), quote_plus(DB_PWD), DB_HOST, DB_PORT, DB_SCHEMA
)
client = MongoClient(conn_url)

In [ ]:
completed_items = list(client.get_database(DB_SCHEMA)["projects_images"].find({ "status": 2 }))
len(completed_items)

In [ ]:
annotations_info = []
for image in tqdm(completed_items):
    filename = image["filename"]
    original_image = Image.open(os.path.join(BASE_PATH, filename)).convert("RGB")
    
    for shape in image["shapes"]:
        label = shape["categories"][0]
        cords = shape["points"]
        x1 = cords[0]
        x3 = cords[2]
        try:
            crop_img = original_image.crop((x1[0], x1[1], x3[0], x3[1]))
        except ValueError as ex:
            print("Cagada en {} - {}".format(str(image["_id"]), shape))
        
        crop_img_name = "{}.jpeg".format(uuid4().hex[:12])
        save_path = os.path.join(EXPORT_PATH, crop_img_name)
        crop_img.save(save_path)
        annotations_info.append({
            "label": label,
            "image": save_path
        })

In [ ]:
json.dump(annotations_info, open(os.path.join(EXPORT_PATH, "metadata.json"), "w", encoding="utf-8"))